# 05 - Bronze Common Helpers

            Shared Unity Catalog-compatible helpers for audit, file-level
            idempotency, Delta transactions, lineage, schema logging, and DQ
            metrics. Other Bronze notebooks load this notebook with `%run`.

In [ ]:
import hashlib
import re
from datetime import datetime, timezone

from delta.tables import DeltaTable
from pyspark.sql import Row
from pyspark.sql import functions as F
from pyspark.sql import types as T


IDENTIFIER_PATTERN = re.compile(r"[A-Za-z_][A-Za-z0-9_]*")

AUDIT_SCHEMA = T.StructType(
    [
        T.StructField("pipeline_run_id", T.StringType(), False),
        T.StructField("pipeline_name", T.StringType(), False),
        T.StructField("batch_id", T.StringType(), True),
        T.StructField("start_ts", T.TimestampType(), False),
        T.StructField("end_ts", T.TimestampType(), True),
        T.StructField("status", T.StringType(), False),
        T.StructField("rows_read", T.LongType(), True),
        T.StructField("rows_written", T.LongType(), True),
        T.StructField("rows_quarantined", T.LongType(), True),
        T.StructField("source_paths", T.ArrayType(T.StringType()), True),
        T.StructField("notebook_job_run_id", T.StringType(), True),
        T.StructField("error_message", T.StringType(), True),
    ]
)

FILE_SCHEMA = T.StructType(
    [
        T.StructField("source_file", T.StringType(), False),
        T.StructField("file_size", T.LongType(), False),
        T.StructField("modification_time", T.TimestampType(), False),
    ]
)

DQ_SCHEMA = T.StructType(
    [
        T.StructField("quality_date", T.DateType(), False),
        T.StructField("source_name", T.StringType(), False),
        T.StructField("pipeline_stage", T.StringType(), False),
        T.StructField("quality_status", T.StringType(), False),
        T.StructField("error_code", T.StringType(), True),
        T.StructField("record_count", T.LongType(), False),
        T.StructField("pipeline_run_id", T.StringType(), False),
        T.StructField("recorded_ts", T.TimestampType(), False),
    ]
)

SCHEMA_LOG_SCHEMA = T.StructType(
    [
        T.StructField("source_name", T.StringType(), False),
        T.StructField("detected_ts", T.TimestampType(), False),
        T.StructField("source_file", T.StringType(), True),
        T.StructField("batch_id", T.StringType(), True),
        T.StructField("schema_version", T.StringType(), False),
        T.StructField("previous_schema_json", T.StringType(), True),
        T.StructField("current_schema_json", T.StringType(), False),
        T.StructField("change_type", T.StringType(), False),
        T.StructField("pipeline_run_id", T.StringType(), True),
    ]
)


def validate_identifier(value: str, label: str) -> str:
    if not IDENTIFIER_PATTERN.fullmatch(value):
        raise ValueError(f"Invalid {label}: {value!r}")
    return value


def recursive_files(root: str, extensions: set[str] | None = None) -> list[Row]:
    results = []
    stack = [root.rstrip("/")]
    while stack:
        current = stack.pop()
        for item in dbutils.fs.ls(current):
            if item.isDir():
                stack.append(item.path.rstrip("/"))
                continue
            lower_name = item.name.lower()
            if lower_name.startswith("_") or lower_name.startswith("."):
                continue
            if extensions and not any(lower_name.endswith(ext) for ext in extensions):
                continue
            results.append(
                Row(
                    source_file=item.path,
                    file_size=int(item.size),
                    modification_time=datetime.fromtimestamp(
                        item.modificationTime / 1000, timezone.utc
                    ),
                )
            )
    return results


def pending_files(
    catalog: str,
    source_name: str,
    source_root: str,
    extensions: set[str] | None = None,
):
    files = recursive_files(source_root, extensions)
    all_files = spark.createDataFrame(files, FILE_SCHEMA)
    if not files:
        return all_files

    processed = (
        spark.table(f"{catalog}.control.processed_files")
        .filter(
            (F.col("source_name") == source_name)
            & (F.col("status") == "SUCCESS")
        )
        .select("source_file", "file_size", "modification_time")
        .dropDuplicates()
    )
    return all_files.join(
        processed,
        ["source_file", "file_size", "modification_time"],
        "left_anti",
    )


def filter_unprocessed_stream_batch(batch_df, catalog: str, source_name: str):
    batch_files = batch_df.select(
        "source_file", "file_size", "modification_time"
    ).dropDuplicates()
    processed = (
        spark.table(f"{catalog}.control.processed_files")
        .filter(
            (F.col("source_name") == source_name)
            & (F.col("status") == "SUCCESS")
        )
        .select("source_file", "file_size", "modification_time")
        .dropDuplicates()
    )
    pending = batch_files.join(
        processed,
        ["source_file", "file_size", "modification_time"],
        "left_anti",
    )
    return batch_df.join(
        F.broadcast(pending),
        ["source_file", "file_size", "modification_time"],
        "inner",
    ), pending


def batch_token(file_df) -> tuple[str, int]:
    file_rows = file_df.orderBy(
        "source_file", "file_size", "modification_time"
    ).collect()
    material = "|".join(
        f"{row.source_file}:{row.file_size}:{row.modification_time.isoformat()}"
        for row in file_rows
    )
    digest = hashlib.sha256(material.encode("utf-8")).hexdigest()
    return digest, int(digest[:15], 16)


def add_lineage(df, pipeline_run_id: str):
    return (
        df.withColumn(
            "batch_id",
            F.regexp_extract("source_file", r"batch_id=([^/]+)", 1),
        )
        .withColumn(
            "load_date",
            F.to_date(F.regexp_extract("source_file", r"load_date=([^/]+)", 1)),
        )
        .withColumn("ingest_ts", F.current_timestamp())
        .withColumn("pipeline_run_id", F.lit(pipeline_run_id))
    )


def append_delta(
    df,
    table_name: str,
    transaction_app_id: str,
    transaction_version: int,
    merge_schema: bool = False,
) -> int:
    row_count = df.count()
    if row_count == 0:
        return 0
    writer = (
        df.write.format("delta")
        .mode("append")
        .option("txnAppId", transaction_app_id)
        .option("txnVersion", transaction_version)
    )
    if merge_schema:
        writer = writer.option("mergeSchema", "true")
    writer.saveAsTable(table_name)
    return row_count


def upsert_audit(
    catalog: str,
    pipeline_run_id: str,
    pipeline_name: str,
    batch_id: str | None,
    start_ts: datetime,
    status: str,
    source_paths: list[str],
    notebook_job_run_id: str,
    end_ts: datetime | None = None,
    rows_read: int | None = None,
    rows_written: int | None = None,
    rows_quarantined: int | None = None,
    error_message: str | None = None,
) -> None:
    row = Row(
        pipeline_run_id=pipeline_run_id,
        pipeline_name=pipeline_name,
        batch_id=batch_id,
        start_ts=start_ts,
        end_ts=end_ts,
        status=status,
        rows_read=rows_read,
        rows_written=rows_written,
        rows_quarantined=rows_quarantined,
        source_paths=source_paths,
        notebook_job_run_id=notebook_job_run_id,
        error_message=error_message,
    )
    source = spark.createDataFrame([row], AUDIT_SCHEMA)
    target = DeltaTable.forName(spark, f"{catalog}.control.audit_pipeline_runs")
    (
        target.alias("target")
        .merge(
            source.alias("source"),
            "target.pipeline_run_id = source.pipeline_run_id",
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )


def mark_files_success(
    catalog: str,
    source_name: str,
    files_df,
    pipeline_run_id: str,
) -> None:
    source = (
        files_df.select("source_file", "file_size", "modification_time")
        .dropDuplicates()
        .withColumn("source_name", F.lit(source_name))
        .withColumn("content_hash", F.lit(None).cast("string"))
        .withColumn(
            "batch_id",
            F.regexp_extract("source_file", r"batch_id=([^/]+)", 1),
        )
        .withColumn("first_seen_ts", F.current_timestamp())
        .withColumn("processed_ts", F.current_timestamp())
        .withColumn("pipeline_run_id", F.lit(pipeline_run_id))
        .withColumn("status", F.lit("SUCCESS"))
        .withColumn("rows_read", F.lit(None).cast("long"))
        .withColumn("rows_written", F.lit(None).cast("long"))
        .withColumn("rows_quarantined", F.lit(None).cast("long"))
        .withColumn("error_message", F.lit(None).cast("string"))
        .select(
            "source_name",
            "source_file",
            "file_size",
            "modification_time",
            "content_hash",
            "batch_id",
            "first_seen_ts",
            "processed_ts",
            "pipeline_run_id",
            "status",
            "rows_read",
            "rows_written",
            "rows_quarantined",
            "error_message",
        )
    )
    target = DeltaTable.forName(spark, f"{catalog}.control.processed_files")
    (
        target.alias("target")
        .merge(
            source.alias("source"),
            """
            target.source_name = source.source_name
            AND target.source_file = source.source_file
            AND target.file_size = source.file_size
            AND target.modification_time = source.modification_time
            """,
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )


def write_dq_metrics(
    catalog: str,
    source_name: str,
    pipeline_run_id: str,
    transaction_version: int,
    valid_count: int,
    error_counts: list[Row],
) -> None:
    now = datetime.now(timezone.utc)
    rows = [
        Row(
            quality_date=now.date(),
            source_name=source_name,
            pipeline_stage="bronze",
            quality_status="valid",
            error_code=None,
            record_count=int(valid_count),
            pipeline_run_id=pipeline_run_id,
            recorded_ts=now,
        )
    ]
    rows.extend(
        Row(
            quality_date=now.date(),
            source_name=source_name,
            pipeline_stage="bronze",
            quality_status="error",
            error_code=row.error_code,
            record_count=int(row["count"]),
            pipeline_run_id=pipeline_run_id,
            recorded_ts=now,
        )
        for row in error_counts
    )
    metrics = spark.createDataFrame(rows, DQ_SCHEMA)
    append_delta(
        metrics,
        f"{catalog}.control.data_quality_results",
        f"{source_name}_bronze_dq",
        transaction_version,
    )


def log_schema_version(
    catalog: str,
    source_name: str,
    schema_json: str,
    files_df,
    pipeline_run_id: str,
) -> str:
    version = hashlib.sha256(schema_json.encode("utf-8")).hexdigest()
    source_file = files_df.orderBy("source_file").first()["source_file"]
    batch_id = re.search(r"batch_id=([^/]+)", source_file)
    source = spark.createDataFrame(
        [
            Row(
                source_name=source_name,
                detected_ts=datetime.now(timezone.utc),
                source_file=source_file,
                batch_id=batch_id.group(1) if batch_id else None,
                schema_version=version,
                previous_schema_json=None,
                current_schema_json=schema_json,
                change_type="observed",
                pipeline_run_id=pipeline_run_id,
            )
        ],
        SCHEMA_LOG_SCHEMA,
    )
    target = DeltaTable.forName(spark, f"{catalog}.control.schema_change_log")
    (
        target.alias("target")
        .merge(
            source.alias("source"),
            """
            target.source_name = source.source_name
            AND target.schema_version = source.schema_version
            """,
        )
        .whenNotMatchedInsertAll()
        .execute()
    )
    return version